In [ ]:
import sys
!git clone -b text-detection-model https://github.com/duckysmacky/cogniflex.git
%cd cogniflex
%cd /kaggle/working/cogniflex
!pwd
!ls model
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".."))) 
sys.path.insert(0, './model')

In [ ]:
import torch.nn as nn
import os
from PIL import Image
from torchvision import transforms
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
import sys
from sklearn.metrics import f1_score

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), ".."))) 
from utils.text_utils.text_preprocess_torch import AIDetectionDataset, RoBERTaDetector

/Users/iaroslav/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/iaroslav/Library/Python/3.9/lib/python/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [3]:
BATCH_SIZE = 16
MODEL_NAME = 'roberta-base'

In [8]:
# dataset = pd.read_csv('/kaggle/input/datasets/...')
dataset = pd.read_csv('AI_Human.csv')
dataset = dataset[::5]

import gc
gc.collect()

data, labels = dataset['text'], dataset['generated'].astype(int) #1 = ai, 0 = human
data_train, data_test, labels_train, labels_test = train_test_split(data, labels, test_size=0.2, random_state=42)
print(len(labels), len(data))
print(labels[labels == 1].sum(), labels[labels == 0].count())

97447 97447
36016 61431


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


train_dataset = AIDetectionDataset(
    data_train, labels_train, tokenizer
)

test_dataset = AIDetectionDataset(
    data_test, labels_test, tokenizer
)

/Users/iaroslav/Library/Python/3.9/lib/python/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,         
    num_workers=2       
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False   
)

In [ ]:
"""
Далее происходит подготовка текстов и fine tuning модели RoBERTa
Подготовка текста - просто разделение его на токены и группировка по батчам 
с помощью AIDetectionDataset

Векторизация происходит автоматически в первых слоях RoBERTa
"""


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
roberta = RoBERTaDetector(MODEL_NAME, 0.2) #custom model from text_utils


optimizer = torch.optim.AdamW(roberta.parameters(), lr=2e-5)
weights = torch.tensor([1.0, 61431/36016], dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.5)

roberta = roberta.to(device)
print(f'device: {device}')

In [ ]:
def train_model_with_early_stopping(num_epochs=5):

    best_val_loss = float('inf')
    patience = 2
    epochs_no_improve = 0
    save_path = 'best_roberta_fc.pth'

    for epoch in range(num_epochs):

        print(f"Epoch {epoch+1}")
        roberta.train()
        train_loss = 0
        train_batches = 0

        for texts, labels in train_loader:
            texts = texts.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = roberta(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_batches += 1

        avg_train_loss = train_loss/train_batches
        print(f'Train loss: {avg_train_loss:.4f}')

        roberta.eval()
        val_loss = 0

        pred_epoch = []
        labels_epoch = []

        with torch.no_grad():
            for texts, labels in test_loader:
                texts = texts.to(device)
                labels = labels.to(device)

                outputs = roberta(texts)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                pred = torch.argmax(outputs, dim=1)
                pred_epoch.extend(pred.detach().cpu().numpy())
                labels_epoch.extend(labels.cpu().numpy())

        avg_val_loss = val_loss/len(test_loader)
        f1score = f1_score(labels_epoch, pred_epoch)

        print(f"Val_loss: {avg_val_loss:.4f}")
        print(f"f1score: {f1score:.4f}")

        #early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0

            torch.save(roberta.state_dict(), save_path)
            print(f'Model saved at epoch {epoch+1}')
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print('There is no improvement')
            break

        scheduler.step()

In [ ]:
train_model_with_early_stopping(2)